In [1]:
import duckdb as db
import pandas as pd

In [2]:
visimages = db.connect("./visimages.db")

In [3]:
visimages.sql("SHOW TABLES").show()

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ author          │
│ contribution    │
│ figure          │
│ figure_property │
│ institution     │
│ paper           │
│ residence       │
└─────────────────┘



In [4]:
visimages.sql("SELECT * FROM paper").show()

┌────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────┬──────────────────┬───────────────────────────────────────────────────────────────────────────────────────┬──────────┬───────────┐
│     id     │                                                         title                                                          │                    doi                     │ publication_date │                                        oa_url                                         │ pdf_path │  inst_id  │
│   int64    │                                                        varchar                                                         │                  varchar                   │       date       │                                        varchar                                        │ varchar  │   int64   │
├────────────┼─────────────────────────────────────────────────────

In [5]:
metadata = pd.read_csv("./metadata.csv",header=None,names=["journal","title","doi","link"])

In [6]:
metadata

,journal,title,doi,link
0,Vis,Interdisciplinary visualization: lessons learn...,10.0000/00000002,http://dl.acm.org/citation.cfm?id=949606&CFID=...
1,Vis,Surface representations of two- and three-dime...,10.1109/VISUAL.1990.146359,http://dx.doi.org/10.1109/VISUAL.1990.146359
2,Vis,FAST: a multi-processed environment for visual...,10.1109/VISUAL.1990.146360,http://dx.doi.org/10.1109/VISUAL.1990.146360
3,Vis,The VIS-5D system for easy interactive visuali...,10.1109/VISUAL.1990.146361,http://dx.doi.org/10.1109/VISUAL.1990.146361
4,Vis,A procedural interface for volume rendering,10.1109/VISUAL.1990.146362,http://dx.doi.org/10.1109/VISUAL.1990.146362
...,...,...,...,...
3096,VAST,RegressionExplorer: Interactive Exploration of...,10.1109/TVCG.2018.2865043,http://dx.doi.org/10.1109/TVCG.2018.2865043
3097,VAST,Seq2Seq-Vis: A Visual Debugging Tool for Seque...,10.1109/TVCG.2018.2865044,http://dx.doi.org/10.1109/TVCG.2018.2865044
3098,VAST,"SIRIUS: Dual, Symmetric, Interactive Dimension...",10.1109/TVCG.2018.2865047,http://dx.doi.org/10.1109/TVCG.2018.2865047
3099,VAST,MotionRugs: Visualizing Collective Trends in S...,10.1109/TVCG.2018.2865049,http://dx.doi.org/10.1109/TVCG.2018.2865049


In [7]:
import numpy as np

In [8]:
dois = metadata[metadata.link.str.contains("doi.org/")].copy()
dois["metadata_index"] = dois.index
dois["database_index"]=np.arange(dois.shape[0])

In [9]:
dois

,journal,title,doi,link,metadata_index,database_index
1,Vis,Surface representations of two- and three-dime...,10.1109/VISUAL.1990.146359,http://dx.doi.org/10.1109/VISUAL.1990.146359,1,0
2,Vis,FAST: a multi-processed environment for visual...,10.1109/VISUAL.1990.146360,http://dx.doi.org/10.1109/VISUAL.1990.146360,2,1
3,Vis,The VIS-5D system for easy interactive visuali...,10.1109/VISUAL.1990.146361,http://dx.doi.org/10.1109/VISUAL.1990.146361,3,2
4,Vis,A procedural interface for volume rendering,10.1109/VISUAL.1990.146362,http://dx.doi.org/10.1109/VISUAL.1990.146362,4,3
5,Vis,Techniques for the interactive visualization o...,10.1109/VISUAL.1990.146363,http://dx.doi.org/10.1109/VISUAL.1990.146363,5,4
...,...,...,...,...,...,...
3096,VAST,RegressionExplorer: Interactive Exploration of...,10.1109/TVCG.2018.2865043,http://dx.doi.org/10.1109/TVCG.2018.2865043,3096,3048
3097,VAST,Seq2Seq-Vis: A Visual Debugging Tool for Seque...,10.1109/TVCG.2018.2865044,http://dx.doi.org/10.1109/TVCG.2018.2865044,3097,3049
3098,VAST,"SIRIUS: Dual, Symmetric, Interactive Dimension...",10.1109/TVCG.2018.2865047,http://dx.doi.org/10.1109/TVCG.2018.2865047,3098,3050
3099,VAST,MotionRugs: Visualizing Collective Trends in S...,10.1109/TVCG.2018.2865049,http://dx.doi.org/10.1109/TVCG.2018.2865049,3099,3051


In [10]:
import json

In [11]:
from pathlib import Path

In [12]:
annotation_data = json.loads(Path("./annotation.json").read_text())

In [13]:
original_db = db.connect("./original.db")

In [14]:
original_db.sql("SELECT * FROM figure").show()

CatalogException: Catalog Error: Table with name figure does not exist!
Did you mean "pg_attrdef"?

In [15]:
# read an annotation entry,
# find the row in the metadata table it corresponds with, which is 1 the annotation value
# follow the structure for the figure table in the original db

# figure entry needs 3 things, paper id, a local path (will ignore for now), and then a server path following this pattern
# https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/zhiyang_visimages_subfigures/subfigures/1021_0000_0.png
# but note that on the server the number is left padded by zeros which makes things kind of weird




In [16]:
# it appears that I skip the results in here that don't have a visualization associated with them
# otherwise format is {paper_id}_{image_id:left aligned, 4 0s padded}_{bbox_list_index:no padding or alignment}

annotation_paper_id = 1021
annotation_data[str(annotation_paper_id)]



# iterate over the images for this record


metadata_row = metadata.iloc[annotation_paper_id]
metadata_row

metadata_row.title

# keep in mind that for now the visimages database doesn't include things that didn't have a nice doi, so we can't map directly to the table entry number, we have to base on titles



visimages_entry = visimages.sql(f"SELECT * FROM paper WHERE title like '{metadata_row.title}'")

so the process should be like this,
for each paper key we
then we also get a row from the metadata
we then also get the paper_id using a WHERE clause on that metadata rows' title
then we 
iterate over the images
each one we check whether there's a vbox, if so we iterate over them
then we construct a subfigure url, this contains the paper id, the image id, and then the index of the bbox 
for each one we store the subfigure url, the subfigure id, the paper id, and also additional attributes that may be of interest to the subfigure (chart type associated with it), caption information, 

In [17]:
visimages.sql("SELECT * FROM paper WHERE title like 'Keynote*'").show()

┌───────┬─────────┬─────────┬──────────────────┬─────────┬──────────┬─────────┐
│  id   │  title  │   doi   │ publication_date │ oa_url  │ pdf_path │ inst_id │
│ int64 │ varchar │ varchar │       date       │ varchar │ varchar  │  int64  │
├───────┴─────────┴─────────┴──────────────────┴─────────┴──────────┴─────────┤
│                                   0 rows                                    │
└─────────────────────────────────────────────────────────────────────────────┘



In [18]:
import numpy as np

In [19]:
dois

,journal,title,doi,link,metadata_index,database_index
1,Vis,Surface representations of two- and three-dime...,10.1109/VISUAL.1990.146359,http://dx.doi.org/10.1109/VISUAL.1990.146359,1,0
2,Vis,FAST: a multi-processed environment for visual...,10.1109/VISUAL.1990.146360,http://dx.doi.org/10.1109/VISUAL.1990.146360,2,1
3,Vis,The VIS-5D system for easy interactive visuali...,10.1109/VISUAL.1990.146361,http://dx.doi.org/10.1109/VISUAL.1990.146361,3,2
4,Vis,A procedural interface for volume rendering,10.1109/VISUAL.1990.146362,http://dx.doi.org/10.1109/VISUAL.1990.146362,4,3
5,Vis,Techniques for the interactive visualization o...,10.1109/VISUAL.1990.146363,http://dx.doi.org/10.1109/VISUAL.1990.146363,5,4
...,...,...,...,...,...,...
3096,VAST,RegressionExplorer: Interactive Exploration of...,10.1109/TVCG.2018.2865043,http://dx.doi.org/10.1109/TVCG.2018.2865043,3096,3048
3097,VAST,Seq2Seq-Vis: A Visual Debugging Tool for Seque...,10.1109/TVCG.2018.2865044,http://dx.doi.org/10.1109/TVCG.2018.2865044,3097,3049
3098,VAST,"SIRIUS: Dual, Symmetric, Interactive Dimension...",10.1109/TVCG.2018.2865047,http://dx.doi.org/10.1109/TVCG.2018.2865047,3098,3050
3099,VAST,MotionRugs: Visualizing Collective Trends in S...,10.1109/TVCG.2018.2865049,http://dx.doi.org/10.1109/TVCG.2018.2865049,3099,3051


In [20]:
index_map= {r.metadata_index:r.database_index for i,r in dois.iterrows()}

In [21]:
index_map

{1: 0,
 2: 1,
 3: 2,
 4: 3,
 5: 4,
 6: 5,
 7: 6,
 8: 7,
 9: 8,
 10: 9,
 11: 10,
 12: 11,
 13: 12,
 14: 13,
 15: 14,
 16: 15,
 17: 16,
 18: 17,
 19: 18,
 20: 19,
 21: 20,
 22: 21,
 23: 22,
 24: 23,
 25: 24,
 26: 25,
 27: 26,
 28: 27,
 29: 28,
 30: 29,
 31: 30,
 32: 31,
 33: 32,
 34: 33,
 35: 34,
 36: 35,
 37: 36,
 38: 37,
 39: 38,
 40: 39,
 41: 40,
 42: 41,
 43: 42,
 44: 43,
 45: 44,
 46: 45,
 47: 46,
 48: 47,
 49: 48,
 50: 49,
 51: 50,
 52: 51,
 53: 52,
 54: 53,
 55: 54,
 56: 55,
 57: 56,
 58: 57,
 59: 58,
 60: 59,
 61: 60,
 62: 61,
 63: 62,
 64: 63,
 65: 64,
 66: 65,
 67: 66,
 68: 67,
 69: 68,
 70: 69,
 71: 70,
 72: 71,
 73: 72,
 74: 73,
 75: 74,
 76: 75,
 77: 76,
 78: 77,
 79: 78,
 80: 79,
 81: 80,
 82: 81,
 83: 82,
 84: 83,
 85: 84,
 86: 85,
 87: 86,
 88: 87,
 89: 88,
 90: 89,
 91: 90,
 92: 91,
 93: 92,
 94: 93,
 95: 94,
 96: 95,
 97: 96,
 98: 97,
 99: 98,
 100: 99,
 101: 100,
 102: 101,
 103: 102,
 104: 103,
 105: 104,
 106: 105,
 107: 106,
 108: 107,
 109: 108,
 110: 109,
 111: 11

In [22]:
import tqdm

In [23]:
# it appears that I skip the results in here that don't have a visualization associated with them
# otherwise format is {paper_id}_{image_id:left aligned, 4 0s padded}_{bbox_list_index:no padding or alignment}
results = []
stop = False
for annotation_paper_id in tqdm.tqdm(annotation_data):
    # iterate over the images for this record
    # only expect one result so take the 0th
    doi_index = index_map.get(int(annotation_paper_id),-1)
    if doi_index == -1:
        # print("missing",annotation_paper_id)
        continue
    # print(doi_index)
    dois_row = dois.iloc[doi_index]
    
    
    
    paper_title = dois_row.title
    # print(paper_title)
    # get the row from the database this corresponds to
    database_index = int(dois_row.database_index)
    # print(database_index,paper_title)
    visimages_entry = visimages.sql(f"SELECT * from paper OFFSET {database_index} LIMIT 1")

    paper_annotation = annotation_data[str(annotation_paper_id)]
    for im in paper_annotation:
        # keys are 
        vbbox=im["visualization_bbox"]
        im_id = im["image_id"]
        caption = im["caption"]
        if len(vbbox)==0:
            continue
        else:
            # iterate over the charts
            index = 0
            
            for chart_type,charts in vbbox.items():
                for c in charts:
                    # add a row to the results for each of these
                    # increment the index
                    url = f"https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/zhiyang_visimages_subfigures/subfigures/{annotation_paper_id}_{im_id:<04d}_{index}.png"
                    index+=1
                    
                    results.append(dict(
                        paper_id=visimages_entry.id.fetchone()[0],
                        server_path=url,
                        caption=caption,
                        chart_type=chart_type
                    ))
            
results

100%|███████████████████████████████████████| 1395/1395 [00:41<00:00, 33.90it/s]


[{'paper_id': 2165915693,
  'server_path': 'https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/zhiyang_visimages_subfigures/subfigures/1021_0000_0.png',
  'caption': 'Figure 2: u, w space diagram',
  'chart_type': 'line_chart'},
 {'paper_id': 2165915693,
  'server_path': 'https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/zhiyang_visimages_subfigures/subfigures/1021_2000_0.png',
  'caption': 'Figure 1: World space and image space',
  'chart_type': 'line_chart'},
 {'paper_id': 2165915693,
  'server_path': 'https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/zhiyang_visimages_subfigures/subfigures/1021_3000_0.png',
  'caption': 'Figure 3: Metric in (u, w) space',
  'chart_type': 'line_chart'},
 {'paper_id': 2165915693,
  'server_path': 'https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/zhiyang_visimages_subfigures/subfigures/1021_3000_1.png',
  'caption': 'Figure 3: Metric in (u, w) space',
  'chart_type': 'line_chart'},
 {'

In [24]:
figures_df = pd.DataFrame(results)

In [25]:
visimages.sql("DROP TABLE figure")

In [26]:
visimages.sql("CREATE TABLE figure AS SELECT * FROM figures_df")

In [27]:
visimages.sql("SELECT * from figure")

┌────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────┐
│  paper_id  │                                                        server_path                                                         │                                                        caption                                                        │         chart_type         │
│   int64    │                                                          varchar                                                           │                                                        varchar                                                        │          varchar           │
├────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [28]:
len(results)

35024

In [29]:
# now working on making things like citation counts and such visible in the paper data
paper_df = visimages.sql("SELECT * FROM paper").df()

In [30]:
paper_df

,id,title,doi,publication_date,oa_url,pdf_path,inst_id
0,3085563693,Surface representations of two- and three-dime...,https://doi.org/10.1109/visual.1990.146359,2002-12-04,None,None,138006243
1,3041520455,FAST: a multi-processed environment for visual...,https://doi.org/10.1109/visual.1990.146360,2002-12-04,https://jaxa.repo.nii.ac.jp/record/43435/files...,None,138006243
2,4237685258,The VIS-5D system for easy interactive visuali...,https://doi.org/10.1109/visual.1990.146361,2002-12-04,None,None,138006243
3,4241306103,A procedural interface for volume rendering,https://doi.org/10.1109/visual.1990.146362,2002-12-04,None,None,138006243
4,4242274056,Techniques for the interactive visualization o...,https://doi.org/10.1109/visual.1990.146363,2002-12-04,https://escholarship.org/content/qt4367m3t4/qt...,None,138006243
...,...,...,...,...,...,...,...
3048,2889730816,RegressionExplorer: Interactive Exploration of...,https://doi.org/10.1109/tvcg.2018.2865043,2018-09-13,https://pure.tue.nl/ws/files/116894719/0846430...,None,138006243
3049,2963123635,Seq2seq-Vis: A Visual Debugging Tool for Seque...,https://doi.org/10.1109/tvcg.2018.2865044,2018-10-17,https://arxiv.org/pdf/1804.09299,None,138006243
3050,2888244365,"SIRIUS: Dual, Symmetric, Interactive Dimension...",https://doi.org/10.1109/tvcg.2018.2865047,2018-08-20,None,None,138006243
3051,2888207115,MotionRugs: Visualizing Collective Trends in S...,https://doi.org/10.1109/tvcg.2018.2865049,2018-08-20,https://kops.uni-konstanz.de/bitstreams/6238ff...,None,138006243


In [31]:
all_open_alex_data = json.loads(Path("./Hear_me_ROR_out_visimages.json").read_text())

In [32]:
res = []
for e in all_open_alex_data:
    ptopic = e.get("primary_topic",-1)
    if ptopic!=-1 and ptopic != None:
        ptopic = ptopic.get("display_name","")
    else:
        ptopic = ""
    res.append(dict(fwci=e["fwci"],cited_by_count=e["cited_by_count"],referenced_works=e["referenced_works_count"],
         primary_topic=ptopic
    )
              )

res

[{'fwci': 6.892,
  'cited_by_count': 59,
  'referenced_works': 10,
  'primary_topic': 'Fluid Dynamics and Turbulent Flows'},
 {'fwci': 5.675,
  'cited_by_count': 41,
  'referenced_works': 11,
  'primary_topic': 'Fluid Dynamics and Vibration Analysis'},
 {'fwci': 2.605,
  'cited_by_count': 28,
  'referenced_works': 1,
  'primary_topic': 'Distributed and Parallel Computing Systems'},
 {'fwci': 1.148,
  'cited_by_count': 3,
  'referenced_works': 8,
  'primary_topic': 'Computer Graphics and Visualization Techniques'},
 {'fwci': 1.531,
  'cited_by_count': 6,
  'referenced_works': 27,
  'primary_topic': 'Computer Graphics and Visualization Techniques'},
 {'fwci': 0.0,
  'cited_by_count': 0,
  'referenced_works': 32,
  'primary_topic': 'Computer Graphics and Visualization Techniques'},
 {'fwci': 2.192,
  'cited_by_count': 26,
  'referenced_works': 9,
  'primary_topic': 'Advanced Vision and Imaging'},
 {'fwci': 0.0,
  'cited_by_count': 0,
  'referenced_works': 3,
  'primary_topic': 'Image Proc

In [33]:
paper_citation_df = pd.DataFrame(res)

In [34]:
all_paper_df = pd.concat([paper_df,paper_citation_df],axis=1)

In [35]:
visimages.sql("DROP TABLE paper")

In [36]:
visimages.sql("CREATE TABLE paper AS SELECT * FROM all_paper_df")

In [37]:
visimages.execute("EXPORT DATABASE 'visimages_parquet' (FORMAT parquet)")

In [38]:
results[8000]

{'paper_id': 2170910862,
 'server_path': 'https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/zhiyang_visimages_subfigures/subfigures/1968_8000_1.png',
 'caption': 'Fig. 10. Visualization of the Pollen data set: (a) Scatterplot; (b) Con- ventional parallel coordinates; (c) Brightness used to represent density of proﬁles; (d) Interactive Parallel Coordinates Frequency Plots with a brightness gradient; (e) Parallel coordinate plot with line stacks; (f) The central stacks colored by height; (g) Scatterplot of the subset of points in (f).',
 'chart_type': 'parallel_coordinate'}

In [39]:
visimages.sql("SELECT * FROM paper WHERE id = 2170910862").show()

┌────────────┬──────────────────────────────────────────────────┬───────────────────────────────────────┬─────────────────────┬─────────┬──────────┬───────────┬────────┬────────────────┬──────────────────┬──────────────────────────────────┐
│     id     │                      title                       │                  doi                  │  publication_date   │ oa_url  │ pdf_path │  inst_id  │  fwci  │ cited_by_count │ referenced_works │          primary_topic           │
│   int64    │                     varchar                      │                varchar                │      timestamp      │ varchar │  int32   │   int64   │ double │     int64      │      int64       │             varchar              │
├────────────┼──────────────────────────────────────────────────┼───────────────────────────────────────┼─────────────────────┼─────────┼──────────┼───────────┼────────┼────────────────┼──────────────────┼──────────────────────────────────┤
│ 2170910862 │ Stacking Graphic Elem

In [40]:
results[0]

{'paper_id': 2165915693,
 'server_path': 'https://data.cyverse.org/dav-anon/iplant/home/carolinarr/vis-sieve/zhiyang_visimages_subfigures/subfigures/1021_0000_0.png',
 'caption': 'Figure 2: u, w space diagram',
 'chart_type': 'line_chart'}

In [41]:
metadata.iloc[1028]

journal                                              InfoVis
title       Multiscale Visualization of Small World Networks
doi                              10.1109/INFVIS.2003.1249011
link       http://doi.ieeecomputersociety.org/10.1109/INF...
Name: 1028, dtype: object

In [42]:
annotation_paper_id

'938'

In [43]:
dois[dois.metadata_index == 3060]

,journal,title,doi,link,metadata_index,database_index
3060,VAST,iForest: Interpreting Random Forests via Visua...,10.1109/TVCG.2018.2864475,http://dx.doi.org/10.1109/TVCG.2018.2864475,3060,3012


In [46]:
visimages.close()